# Problem Statement: Generative Search System for Insurance Policy Documents

## Objective
The objective of this project is to build a robust generative search system capable of effectively and accurately answering questions from various insurance policy documents. The system will leverage advanced natural language processing (NLP) techniques and frameworks such as LangChain or LlamaIndex to retrieve and generate relevant answers from a corpus of insurance policy documents.

## Project Scope
1. **Data Collection**: Gather a diverse set of insurance policy documents. These documents can include health insurance policies, life insurance policies, auto insurance policies, and home insurance policies.

2. **Data Preprocessing**: Preprocess the collected documents to ensure they are in a suitable format for text retrieval and generation. This may include cleaning the text, tokenization, and converting documents into a consistent structure.

3. **Framework Selection**: Choose between LangChain and LlamaIndex as the primary framework for building the generative search system. Both frameworks offer powerful tools for text retrieval and generation.

4. **Model Training and Fine-Tuning**: If necessary, fine-tune pre-trained language models on the insurance policy documents to improve the accuracy and relevance of generated answers.

# **Step 1: Import the necessary Libraries**

In [1]:
!pip install llama-index

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.4/327.4 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.2 MB/s eta 0:00:00


In [2]:
# Install additional supporting libraries as required

**Import Necessary Modules:**

*   fitz from PyMuPDF for reading PDF files.
*   SimpleDocument and DocumentIndex from llama_index.
*   os for handling file paths
*   tabulate for showing outputs in table

In [3]:
# Install OpenAI, LlamaIndex, pymupdf
!pip install -U -qq llama-index openai

In [4]:
!pip install pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 28.6 MB/s eta 0:00:00


In [5]:
!pip install tabulate

In [6]:
# Importing the libraries
from llama_index.core import Document, GPTVectorStoreIndex, Settings, VectorStoreIndex, SimpleDirectoryReader, StorageContext
from concurrent.futures import ThreadPoolExecutor, as_completed
from llama_index.core.tools import QueryEngineTool
from llama_index.core.llms import ChatMessage
from llama_index.llms.openai import OpenAI
from transformers import GPT2Tokenizer
from nltk.corpus import stopwords
from collections import Counter
from tabulate import tabulate
import llama_index.readers
import llama_index.core
import pandas as pd
import llama_index
import logging
import openai
import fitz  # PyMuPDF
import nltk
import json
import sys
import os
import re

# **Step 2: Mount your Google Drive and Set the API key**

In [7]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [8]:
# Set the API key
filepath = "/content/drive/MyDrive/HelpMate Project/"
with open(filepath + "OPENAI_API_Key.txt", "r") as f:
  openai.api_key = ' '.join(f.readlines())

# **Step 3: Data Loading**

In [9]:
# The following line of code prints a list of all the attributes and methods available in the 'llama_index' module.
print(dir(llama_index))

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'core', 'embeddings', 'llms', 'readers']


In [10]:
# The following line of code prints a list of all the attributes and methods available in the 'llama_index.core' submodule.
print(dir(llama_index.core))

# The following line of code prints a list of all the attributes and methods available in the 'llama_index.readers' submodule.
print(dir(llama_index.readers))

['BaseCallbackHandler', 'BasePromptTemplate', 'Callable', 'ChatPromptTemplate', 'ComposableGraph', 'Document', 'DocumentSummaryIndex', 'GPTDocumentSummaryIndex', 'GPTKeywordTableIndex', 'GPTListIndex', 'GPTRAKEKeywordTableIndex', 'GPTSimpleKeywordTableIndex', 'GPTTreeIndex', 'GPTVectorStoreIndex', 'IndexStructType', 'KeywordTableIndex', 'KnowledgeGraphIndex', 'ListIndex', 'MockEmbedding', 'NullHandler', 'Optional', 'Prompt', 'PromptHelper', 'PromptTemplate', 'PropertyGraphIndex', 'QueryBundle', 'RAKEKeywordTableIndex', 'Response', 'SQLContextBuilder', 'SQLDatabase', 'SQLDocumentContextBuilder', 'SelectorPromptTemplate', 'ServiceContext', 'Settings', 'SimpleDirectoryReader', 'SimpleKeywordTableIndex', 'StorageContext', 'SummaryIndex', 'TreeIndex', 'VectorStoreIndex', '__all__', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'async_utils', 'base', 'bridge', 'callbacks', 'chat_engine',

In [11]:
# Define the directory containing your policy documents
directory = "/content/drive/MyDrive/HelpMate Project/Policy_Documents"

In [12]:
# Initialize the tokenizer from the transformers library
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [13]:
# Function to split the text from the documents with the specified max_tokens
def split_text(text, max_tokens=512):
    """
    Split text into chunks based on the token count.

    Args:
    text (str): The input text to be chunked.
    max_tokens (int): The maximum number of tokens per chunk.

    Returns:
    list: A list of text chunks.
    """
    tokens = tokenizer.tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for token in tokens:
        token_length = len(tokenizer.convert_tokens_to_string([token]))
        if current_length + token_length > max_tokens:
            chunks.append(tokenizer.convert_tokens_to_string(current_chunk))
            current_chunk = []
            current_length = 0
        current_chunk.append(token)
        current_length += token_length

    if current_chunk:
        chunks.append(tokenizer.convert_tokens_to_string(current_chunk))

    return chunks

In [14]:
# Function to process sing file out of the documents with the specified max_tokens
def process_single_file(filepath, max_tokens):
    documents = []
    filename = os.path.basename(filepath)
    with fitz.open(filepath) as pdf:
        for page_num, page in enumerate(pdf, start=1):
            text = page.get_text()
            chunks = split_text(text, max_tokens)
            for chunk in chunks:
                metadata = {
                    "file_name": filename,
                    "page_number": page_num
                }
                documents.append(Document(text=chunk, metadata=metadata))
    return documents

In [15]:
# Function to load the documents with the specified max_tokens
def load_documents(directory, max_tokens=512, num_workers=4):
    """
    Load documents from a directory, process them in parallel, and split them into chunks.

    Args:
    directory (str): The directory containing the PDF files.
    max_tokens (int): The maximum number of tokens per chunk.
    num_workers (int): Number of parallel workers.

    Returns:
    list: A list of Document objects.
    """
    documents = []
    pdf_files = [os.path.join(directory, filename) for filename in os.listdir(directory) if filename.endswith(".pdf")]

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        future_to_file = {executor.submit(process_single_file, filepath, max_tokens): filepath for filepath in pdf_files}
        for future in as_completed(future_to_file):
            try:
                documents.extend(future.result())
            except Exception as e:
                print(f"Error processing file {future_to_file[future]}: {e}")

    return documents

In [16]:
# Adjust the max_tokens value as needed
max_tokens = 2048  # Experiment with different values like 256, 512, 1024, etc.

# Load the documents
documents = load_documents(directory, max_tokens, num_workers=4)

# Create an index
index = GPTVectorStoreIndex(documents)

# Convert documents to JSON-serializable format
documents_json = [{"text": doc.text, "metadata": doc.metadata} for doc in documents]

# Define the path to the index file
index_path = '/content/drive/MyDrive/HelpMate Project/index.json'

# Check if the file exists and delete it if it does
if os.path.exists(index_path):
    os.remove(index_path)

# Save the documents and index metadata to JSON
with open(index_path, 'w') as f:
    json.dump(documents_json, f)

print(f"Indexed {len(documents)} documents.")

Indexed 398 documents.


# **Step 4: Building the query engine & Creating a Response Pipeline**

In [17]:
# Check available methods in GPTVectorStoreIndex
print(dir(GPTVectorStoreIndex))

['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_add_nodes_to_index', '_aget_node_with_embedding', '_async_add_nodes_to_index', '_build_index_from_nodes', '_delete_node', '_get_node_with_embedding', '_insert', '_is_protocol', 'as_chat_engine', 'as_query_engine', 'as_retriever', 'build_index_from_nodes', 'delete', 'delete_nodes', 'delete_ref_doc', 'docstore', 'from_documents', 'from_vector_store', 'index_id', 'index_struct', 'index_struct_cls', 'insert', 'insert_nodes', 'ref_doc_info', 'refresh', 'refresh_ref_docs', 'service_context', 'set_index_id', 'storage_context', 's

In [18]:
# Load the documents from JSON
with open(index_path, 'r') as f:
    documents_json = json.load(f)

# Reconstruct Document objects
documents = [Document(text=doc["text"], metadata=doc["metadata"]) for doc in documents_json]

# Recreate the index
index = GPTVectorStoreIndex(documents)

print("Index loaded successfully.")

Index loaded successfully.


In [19]:
# Function to print text with a border
def print_with_border(text):
    border = "#" * (len(max(text.split('\n'), key=len)) + 4)
    print(border)
    for line in text.split('\n'):
        print(f"# {line} #")
    print(border)

In [20]:
nltk.download('stopwords')

# Use nltk's stopwords list
stopwords_set = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [21]:
# Function to extract key terms from user input
def extract_key_terms(user_input):
    """
    Extract key terms from user input using simple term extraction.

    Args:
    user_input (str): The input query provided by the user.

    Returns:
    list: A list of extracted key terms.
    """
    # Simple term extraction by splitting the input on spaces and removing common stopwords
    terms = [word.lower() for word in re.findall(r'\b\w+\b', user_input) if word.lower() not in stopwords_set]
    return terms

In [22]:
# Function to initialize the query engine for answering user questions
def query_response(user_input):
    """
    Generate a response based on user input by querying the query engine and
    retrieving metadata from the source nodes.

    Args:
    user_input (str): The input query provided by the user.

    Returns:
    final_response (str): The final response generated by the query engine, including a
    reference to the source file names and page numbers.
    """

    # Extract key terms from the user input
    key_terms = extract_key_terms(user_input)

    # Create a temporary index with the filtered documents
    temp_index = GPTVectorStoreIndex(documents)
    temp_query_engine = temp_index.as_query_engine()

    # Adjust the query to return more results
    response = temp_query_engine.query(user_input)

    final_response = response.response  # Access the response attribute

    # Debug print to check the structure of the response
    # print("Debug: Response Object:", response)

    # Check if source_nodes attribute exists and if metadata is available
    if hasattr(response, 'source_nodes') and response.source_nodes:
        sources = response.source_nodes  # Access the source_nodes attribute

        # Debug print to check the structure of source nodes
        # print("Debug: Source Nodes:", sources)

        # Filter out sources with empty metadata
        sources_with_metadata = [source for source in sources if source.metadata]

        # Match key terms with document metadata
        relevant_sources = []
        for source in sources_with_metadata:
            file_name = source.metadata.get('file_name', '').lower()
            match_count = sum(term in file_name for term in key_terms)
            if match_count > 0:
                relevant_sources.append((source, match_count))

        # Sort sources by match count in descending order
        relevant_sources.sort(key=lambda x: x[1], reverse=True)

        # Select the top matching source
        if relevant_sources:
            best_match = relevant_sources[0][0]
            source_info = f"Source: {best_match.metadata.get('file_name', 'Unknown')}, Page: {best_match.metadata.get('page_number', 'Unknown')}"
            final_response += f"\n\nReferences:\n{source_info}"

    return final_response


In [23]:
# Function to initialize the conversation with the chatbot
def initialize_conv():
    """
    Initialize a conversation with the user, allowing them to ask questions
    about the policy documents. The user can type 'exit' to end the
    conversation.

    The function continuously prompts the user for input, processes the input
    using the query_response function, and displays the response. The loop
    terminates when the user types 'exit'.
    """
    instructions = (
        "Welcome to the Insurance Policy Document Chatbot!\n"
        "You can ask questions about the policy documents.\n"
        "Type 'exit' to end the conversation."
    )

    # Print instructions with a border
    print_with_border(instructions)

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print_with_border("Goodbye!")
            break

        response = query_response(user_input)
        print(f"Chatbot: {response}")
        print("\n" + "-" * 50 + "\n")  # Print a separator line between each question

In [24]:
# Example usage
initialize_conv()

#####################################################
# Welcome to the Insurance Policy Document Chatbot! #
# You can ask questions about the policy documents. #
# Type 'exit' to end the conversation. #
#####################################################
You: What are the different Guaranteed Benefit Options available under the HDFC Life Sampoorna Jeevan Plan, and how do they affect the Maturity and Survival Benefits?
Chatbot: The different Guaranteed Benefit Options available under the HDFC Life Sampoorna Jeevan Plan are Lump Sum Option, Income Option, Lump Sum with Income Option, and Income with Lump sum Option. 

Under the Lump Sum Option, the Maturity Benefit is 100% of Basic Sum Assured, and the Survival Benefit includes a Guaranteed Maturity Benefit payable on the Policy Maturity Date.

For the Income Option, the Maturity Benefit includes Applicable Bonus and Applicable Terminal Bonus if declared, and the Survival Benefit consists of a Guaranteed Income Benefit equal to 5% of B

# **Step 6: Build a Testing Pipeline**

In [25]:
# List of questions
questions = [
    "How is the Guaranteed Surrender Value (GSV) and Special Surrender Value (SSV) determined for sanchay plus policy?",
    "What are the different fund options available under the HDFC Life Smart Pension Plan, and what are their investment objectives and risk levels?",
    "What are the benefits provided under the different plan options of the HDFC Life Group Poorna Suraksha Policy?",
    "What are the eligibility criteria for an individual to become an Insured Member under the HDFC Life Group Term Life Policy?",
    "What are the benefits provided under the HDFC SurgiCare Plan, and how are they categorized?",
    "What are the exclusions specified in the HDFC SurgiCare Plan, and how do they impact the coverage?",
    "What is the process for filing a claim under the HDFC SurgiCare Plan, and what documents are required?",
    "What are the Maturity and Death Benefits provided under the HDFC Life Sanchay Plus Policy, and how are they calculated?",
    "What are the different Guaranteed Benefit Options available under the HDFC Life Sampoorna Jeevan Plan, and how do they affect the Maturity and Survival Benefits?"
    "What exclusions apply to the Accidental Death Benefit and the Accelerated Critical Illness Benefit?",
    "What is the Free-Look Period for the Master Policyholder and Insured Member, and what are the conditions for cancellation and refund?",
    "What are the definitions and exclusions related to Critical Illnesses covered under the policies?"
]

In [26]:
# Testing pipeline function
def testing_pipeline(questions):
    """
    Conduct a testing pipeline for a series of questions, collecting user
    feedback on the responses provided by the bot.

    Args:
    questions (list): A list of questions to be tested.

    Returns:
    pd.DataFrame: A DataFrame containing the questions, their corresponding
        responses, the references (source document names), the page numbers from the response,
        and the user feedback indicating whether the response was good or bad.
    """
    data = []

    for question in questions:
        response = query_response(question)
        print(f"Question: {question}")
        print(f"Response: {response}")

        # Print a separator line
        print("-" * 80)

        feedback = input("Was the response accurate? (yes/no): ").strip().lower()

        # Print another separator line
        print("=" * 80)

        is_accurate = True if feedback == 'yes' else False

        # Extracting references and page numbers from the response metadata (if available)
        reference = ""
        page_number = ""
        if "References:" in response:
            try:
                references_section = response.split("References:")[1].strip()
                reference_line = references_section.split("\n")[0]
                reference = reference_line.split(", Page:")[0].split("Source:")[1].strip()
                page_number = reference_line.split("Page:")[1].strip().split(",")[0]
            except IndexError:
                reference = "Unknown"
                page_number = "Unknown"

        data.append({
            "Question": question,
            "Response": response,
            "Reference": reference,
            "PageNumber": page_number,
            "Accurate": is_accurate
        })

    df = pd.DataFrame(data)

    return df

In [27]:
# Call the testing pipeline
test_results = testing_pipeline(questions)
print(test_results)

Question: How is the Guaranteed Surrender Value (GSV) and Special Surrender Value (SSV) determined for sanchay plus policy?
Response: The Guaranteed Surrender Value (GSV) for the Sanchay Plus policy is determined based on the applicable GSV factors on Total Premiums Paid at the time of surrender multiplied by the Total Premiums Paid to date. The Special Surrender Value (SSV) is calculated as the present value of future benefits using a discounting factor based on prevailing interest rates, starting from the 5th policy year onwards.

References:
Source: HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf, Page: 10
--------------------------------------------------------------------------------
Was the response accurate? (yes/no): yes
Question: What are the different fund options available under the HDFC Life Smart Pension Plan, and what are their investment objectives and risk levels?
Response: The different fund options available under the HDFC Life Smart Pens

- When a specific document name is included in the user query:
  - The system successfully retrieves the correct document.
  - The system also provides the corresponding page number.

- For more general inquiries:
  - The system provides an answer.
  - However, the output dataframe lacks references and corresponding page numbers.

- Initial testing of the chatbot conversation within the test pipeline appears to be functioning as expected.

- We can now proceed to the subsequent steps.

# Part 2 - Additional Steps

**2.1: Building a custom prompt template**

**Defining the general context for the policy chatbot**

In [28]:
# General context for the policy chatbot RAG pipeline
general_context = """
Insurance policies are legal contracts between an insurer and a policyholder, detailing the claims the insurer must pay. These policies cover various risks, including health, life, property, and liability. Key terms include premiums, deductibles, exclusions, and benefits.

- **Free-Look Period**: Allows policyholders to review and return the policy for a full refund if unsatisfied.
- **Critical Illness Coverage**: Includes specific illnesses and exclusions as defined in the policy.
- **Claims Process**: Requires documentation such as claim forms, proof of loss, and medical records.

Key Points:
- Legal contracts specifying insurer's obligations.
- Covers health, life, property, and liability risks.
- Important terms: premiums, deductibles, exclusions, benefits.
- Free-Look Period for policy review and refund.
- Critical Illness coverage with specific inclusions and exclusions.
- Claims process necessitates documentation.
"""

**Defining the structure of the custom prompt template for the policy chatbot**

In [29]:
# Define the generalized prompt template
custom_prompt_template = """
You are an expert in insurance policies. Your task is to provide accurate, concise, and helpful answers to questions based on the provided context. Use the context to inform your response, and ensure that your answer is clear and easy to understand. If the context does not provide enough information to answer the question, make a note of it in your response.

Context: {context}

Question: {query}

Answer:
a) Summary: {summary}
b) Detailed Explanation: {detailed_explanation}
c) Key Points:
{key_points}
"""

**Defining the helper functions to extract key components from the chatbot output response**

In [30]:
# Function to extract summary section
def extract_summary(response):
    """
    Extract the summary from the response.

    Args:
    response (str): The response text from which to extract the summary.

    Returns:
    str: The extracted summary.
    """
    summary_match = re.search(r'a\)\ Summary:(.*?)(?:b\)\ Detailed Explanation:|$)', response, re.DOTALL)
    if summary_match:
        return summary_match.group(1).strip()
    return "Summary not found."

In [31]:
# Function to extract detailed explanation section
def extract_detailed_explanation(response):
    """
    Extract the detailed explanation from the response.

    Args:
    response (str): The response text from which to extract the detailed explanation.

    Returns:
    str: The extracted detailed explanation.
    """
    detailed_explanation_match = re.search(r'b\)\ Detailed Explanation:(.*?)(?:c\)\ Key Points:|$)', response, re.DOTALL)
    if detailed_explanation_match:
        return detailed_explanation_match.group(1).strip()
    return "Detailed explanation not found."

In [32]:
# Function to extract key points section
def extract_key_points(response):
    """
    Extract the key points from the response.

    Args:
    response (str): The response text from which to extract the key points.

    Returns:
    list: A list of extracted key points.
    """
    key_points_match = re.search(r'c\)\ Key Points:(.*)', response, re.DOTALL)
    if key_points_match:
        key_points_text = key_points_match.group(1).strip()
        key_points = [point.strip() for point in key_points_text.split('- ') if point]
        return key_points
    return ["Key points not found."]

**Changing the logic of quey_response function to accomodate the custom prompt template**

In [33]:
def query_response(user_input):
    """
    Generate a response based on user input by querying the query engine and
    retrieving metadata from the source nodes.

    Args:
    user_input (str): The input query provided by the user.

    Returns:
    tuple: A tuple containing the final response, a list of references, and a list of page numbers.
    """
    # Extract policy name from the user input
    user_input_lower = user_input.lower()

    # Create a temporary index with the filtered documents
    temp_index = GPTVectorStoreIndex(documents)
    temp_query_engine = temp_index.as_query_engine()

    # Create the custom prompt with the general context
    custom_prompt = custom_prompt_template.format(
        context=general_context,
        query=user_input,
        summary="{summary}",  # Placeholder to be filled dynamically
        detailed_explanation="{detailed_explanation}",  # Placeholder to be filled dynamically
        key_points="{key_points}"  # Placeholder to be filled dynamically
    )

    # Generate the initial response using the custom prompt
    response = temp_query_engine.query(custom_prompt)

    # Debug: Print the raw response
    # print("Raw response:", response)

    # Extract the response content
    final_response = response.response  # Access the response attribute

    # Debug: Print the final response content
    # print("Final response content:", final_response)

    # Extract the summary, detailed explanation, and key points from the response
    summary = extract_summary(final_response)
    detailed_explanation = extract_detailed_explanation(final_response)
    key_points = extract_key_points(final_response)

    # Debug: Print extracted sections
    # print("Summary:", summary)
    # print("Detailed Explanation:", detailed_explanation)
    # print("Key Points:", key_points)

    # Format the detailed explanation and key points with tab indentation
    formatted_detailed_explanation = "\n".join([f"\t{line}" for line in detailed_explanation.split('\n')])
    formatted_key_points = "\n".join([f"\t- {point}" for point in key_points])

    # Combine the formatted sections
    formatted_response = f"a). Summary: {summary}\n\nb). Detailed Explanation:\n{formatted_detailed_explanation}\n\nc). Key Points:\n{formatted_key_points}"

    references = []
    page_numbers = []

    # Check if source_nodes attribute exists and if metadata is available
    if hasattr(response, 'source_nodes') and response.source_nodes:
        sources = response.source_nodes  # Access the source_nodes attribute

        # Filter out sources with empty metadata
        sources_with_metadata = [source for source in sources if source.metadata]

        if sources_with_metadata:
            for source in sources_with_metadata:
                references.append(source.metadata.get('file_name', 'Unknown'))
                page_numbers.append(source.metadata.get('page_number', 'Unknown'))

    # Debug: Print references and page numbers
    # print("References:", references)
    # print("Page Numbers:", page_numbers)

    return formatted_response.strip(), references, page_numbers

**Changing the logic of testing_pipeline function to accomodate the custom prompt template**

In [34]:
# Testing pipeline function
def testing_pipeline(questions):
    """
    Conduct a testing pipeline for a series of questions, collecting user
    feedback on the responses provided by the bot.

    Args:
    questions (list): A list of questions to be tested.

    Returns:
    pd.DataFrame: A DataFrame containing the questions, their corresponding
        responses, the references (source document names), the page numbers from the response,
        and the user feedback indicating whether the response was good or bad.
    """
    data = []

    for question in questions:
        response, references, page_numbers = query_response(question)

        # Format the response to start the summary on the next line
        formatted_response = f"Response:\n{'-' * len('Response:')}\n{response}"

        print(f"Question: {question}")
        print(formatted_response)

        # Print a separator line
        print("-" * 80)

        feedback = input("Was the response accurate? (yes/no): ").strip().lower()

        # Print another separator line
        print("=" * 80)

        is_accurate = True if feedback == 'yes' else False

        # Ensure references and page numbers are single values
        reference = references[0] if references else "Unknown"
        page_number = page_numbers[0] if page_numbers else "Unknown"

        data.append({
            "Question": question,
            "Response": response,
            "Reference": reference,
            "PageNumber": page_number,
            "Accurate": is_accurate
        })

    df = pd.DataFrame(data)

    return df

In [35]:
# Call the revised testing pipeline using the custom prompt template
test_results = testing_pipeline(questions)
print(test_results)

Question: How is the Guaranteed Surrender Value (GSV) and Special Surrender Value (SSV) determined for sanchay plus policy?
Response:
---------
a). Summary: The Guaranteed Surrender Value (GSV) for the Sanchay Plus policy is determined based on the total premiums paid at the time of surrender. The Special Surrender Value (SSV) can be higher than the GSV and is calculated using a formula involving future benefits and prevailing interest rates.

b). Detailed Explanation:
	The GSV is calculated as a factor applied to the total premiums paid, excluding any taxes or extra premiums. The SSV can exceed the GSV and is computed differently based on the policy year. In the 2nd to 4th policy years, the SSV is equal to the GSV. From the 5th year onwards, the SSV is determined by the present value of future benefits using a specific formula involving the Annual Income Benefit Amount, premiums paid, and prevailing interest rates.

c). Key Points:
	- GSV is based on total premiums paid at surrender.


In [36]:
# Function to create a chatbot for the custom prompt template
def CustomInsurancePolicyHelper():
    """
    Simulates a chatbot interaction for querying insurance policy documents.
    """
    print("#####################################################")
    print("# Welcome to the Insurance Policy Document Chatbot! #")
    print("# You can ask questions about the policy documents. #")
    print("# Type 'exit' to end the conversation.              #")
    print("#####################################################")

    conversation_data = []

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == 'exit':
            print("############")
            print("# Goodbye! #")
            print("############")
            break

        # Get the response from the query_response function
        response, references, page_numbers = query_response(user_input)

        # Print the chatbot response
        print(f"Chatbot: {response}")
        print("\nReferences:")
        for ref in references:
            print(f"Source: {ref}")
        print("\n" + "-" * 50 + "\n")

        # Append the conversation data
        conversation_data.append({
            "Question": user_input,
            "Answer": response,
            "References": ", ".join(references)
        })

    # Convert the conversation data to a DataFrame
    df = pd.DataFrame(conversation_data)

    # Print the DataFrame as a formatted table using tabulate
    print(tabulate(df, headers='keys', tablefmt='pipe'))

In [37]:
# Example usage
CustomInsurancePolicyHelper()

#####################################################
# Welcome to the Insurance Policy Document Chatbot! #
# You can ask questions about the policy documents. #
# Type 'exit' to end the conversation.              #
#####################################################
You: What are the different Guaranteed Benefit Options available under the HDFC Life Sampoorna Jeevan Plan, and how do they affect the Maturity and Survival Benefits?
Chatbot: a). Summary: The HDFC Life Sampoorna Jeevan Plan offers Guaranteed Benefit Options including Death Benefit, Extra Life Option, Accidental Death Benefit, and Accelerated Critical Illness Option. These benefits impact the Maturity and Survival Benefits based on the specific terms outlined in the policy.

b). Detailed Explanation:
	The Guaranteed Benefit Options under the HDFC Life Sampoorna Jeevan Plan provide different coverage scenarios. The Death Benefit ensures a payout in case of the Scheme Member's demise, with the sum assured being paid out. T

- The output is now correctly formatted according to the specifications set in the custom prompt template.

- We have created a new chatbot **"CustomInsurancePolicyHelper"** specifically for the custom prompt template.

- The output is not exactly 100% true when we look at the references section. We will address this when we implement Retriever Router Query Engine.

Let's proceed to the next section, where we will outline specific use cases for the chatbot we've developed.

**2.2: Recommendations on how to further improve RAG pipeline**

**Use Case 1**: Claim Filing Assistance
User Story: As a policyholder, I want step-by-step guidance on how to file a claim, including the required documents and procedures, so that I can ensure my claim is processed smoothly.

**Implementation**: Create a guided flow that provides detailed instructions based on the user's specific policy and claim type.

In [38]:
# Define the function to print claim filing instructions
def print_claim_filing_instructions(policy_name, claim_type):
    """
    Provide and print step-by-step guidance on how to file a claim for a specific policy and claim type.

    Args:
    policy_name (str): The name of the policy.
    claim_type (str): The type of claim (e.g., health, life, accident).
    """
    # Fetch the claim filing instructions
    response, references, page_numbers = query_response(f"How to file a {claim_type} claim for {policy_name}?")

    # Create a DataFrame for the instructions
    instructions_data = {
        "Policy Name": [policy_name],
        "Claim Type": [claim_type.capitalize()],
        "Instructions": [response],
        "Reference": [references[0] if references else "Unknown"],
        "Page Number": [page_numbers[0] if page_numbers else "Unknown"]
    }
    df = pd.DataFrame(instructions_data)

    # Print the instructions in a formatted table using tabulate
    print(tabulate(df, headers='keys', tablefmt='pipe'))

In [39]:
# Example policy name and claim type
policy_name = "HDFC Life Easy Health Policy"
claim_type = "health"

# Call the function to print claim filing instructions
print_claim_filing_instructions(policy_name, claim_type)

|    | Policy Name                  | Claim Type   | Instructions                                                                                                                                                                                                                                    | Reference                                                   |   Page Number |
|---:|:-----------------------------|:-------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:------------------------------------------------------------|--------------:|
|  0 | HDFC Life Easy Health Policy | Health       | a). Summary: To file a health claim for HDFC Life Easy Health Policy, you need to submit specific documents within 60 days of diagnosis, including a claim form, policy document copy, residence and iden

**Use Case 2**: Recommended Policies
User Story: As a consumer, I want to know about the details about the policies based on my needs

**Implementation**: Integrate a function that takes user inputs and provides policy reccomendations.

In [40]:
# Define the recommend_policies function
def recommend_policies(age, health_condition, coverage_needs, policy_type):
    """
    Recommend the most suitable insurance policies based on user inputs.

    Args:
    age (int): The age of the user.
    health_condition (str): The health condition of the user (e.g., "good", "average", "poor").
    coverage_needs (str): The coverage needs of the user (e.g., "high", "medium", "low").
    policy_type (str): The type of policy the user is interested in (e.g., "health", "life", "pension").

    Returns:
    list: A list of recommended policies with detailed information.
    """
    # Define policy features (basic information)
    policies = [
        {"name": "HDFC Life Easy Health Policy", "type": "health", "coverage": "high", "health_condition": "good"},
        {"name": "HDFC Life Group Poorna Suraksha Policy", "type": "life", "coverage": "medium", "health_condition": "average"},
        {"name": "HDFC Life Group Term Life Policy", "type": "life", "coverage": "high", "health_condition": "good"},
        {"name": "HDFC Life Sampoorna Jeevan Plan", "type": "life", "coverage": "medium", "health_condition": "average"},
        {"name": "HDFC Life Sanchay Plus Policy", "type": "life", "coverage": "high", "health_condition": "good"},
        {"name": "HDFC Life Smart Pension Plan", "type": "pension", "coverage": "medium", "health_condition": "average"},
        {"name": "HDFC Surgicare Plan", "type": "health", "coverage": "low", "health_condition": "poor"},
    ]

    # Filter policies based on user inputs
    filtered_policies = [
        policy for policy in policies
        if policy["type"] == policy_type and
           policy["coverage"] == coverage_needs and
           policy["health_condition"] == health_condition
    ]

    # Fetch detailed information for each recommended policy
    detailed_recommendations = []
    for policy in filtered_policies:
        benefits_response, _, _ = query_response(f"What are the benefits of the {policy['name']}?")
        exclusions_response, _, _ = query_response(f"What are the exclusions of the {policy['name']}?")
        premiums_response, _, _ = query_response(f"What are the premiums of the {policy['name']}?")

        detailed_recommendations.append({
            "Policy Name": policy["name"],
            "Benefits": benefits_response,
            "Exclusions": exclusions_response,
            "Premiums": premiums_response
        })

    return detailed_recommendations

In [41]:
# Define the function to print recommended policies
def print_recommended_policies(recommended_policies):
    """
    Print the recommended insurance policies with detailed information.

    Args:
    recommended_policies (list): A list of dictionaries containing policy information.
    """
    print("Recommended Policies:")
    print("=" * len("Recommended Policies:"))  # Underline with =

    # Create a DataFrame from the recommended policies
    df = pd.DataFrame(recommended_policies)

    # Print the DataFrame as a formatted table using tabulate
    print(tabulate(df, headers='keys', tablefmt='pipe'))

In [42]:
# Example user inputs
age = 35
health_condition = "average"
coverage_needs = "medium"
policy_type = "life"

# Get policy recommendations
recommended_policies = recommend_policies(age, health_condition, coverage_needs, policy_type)

# Print the recommended policies with separators
print_recommended_policies(recommended_policies)

Recommended Policies:
|    | Policy Name                            | Benefits                                                                                                                                                                                                         | Exclusions                                                                                                                                                                                                                                                               | Premiums                                                                                                                                                                                                                       |
|---:|:---------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**Use Case 3**: Premium Calculation
User Story: As a potential policyholder, I want to calculate the premium for a specific insurance policy based on my personal details (e.g., age, health condition), so that I can estimate the cost.

**Implementation**: Integrate a premium calculator that takes user inputs and provides an estimated premium amount.

In [43]:
# Define the premium calculation function
def calculate_premium(policy_name, age, health_condition, sum_assured):
    """
    Calculate the premium for a specific insurance policy based on user inputs.

    Args:
    policy_name (str): The name of the insurance policy.
    age (int): The age of the potential policyholder.
    health_condition (str): The health condition of the potential policyholder (e.g., "good", "average", "poor").
    sum_assured (float): The sum assured for the insurance policy.

    Returns:
    float: The estimated premium amount.
    """
    # Define base premium rates for different policies
    base_premium_rates = {
        "HDFC Life Easy Health Policy": 0.01,  # Example rate: 1% of sum assured
        "HDFC Life Group Poorna Suraksha Policy": 0.015,  # Example rate: 1.5% of sum assured
        "HDFC Life Group Term Life Policy": 0.02,  # Example rate: 2% of sum assured
    }

    # Define health condition multipliers
    health_multipliers = {
        "good": 1.0,
        "average": 1.2,
        "poor": 1.5,
    }

    # Check if the policy name is valid
    if policy_name not in base_premium_rates:
        return "Invalid policy name."

    # Calculate the base premium
    base_premium = base_premium_rates[policy_name] * sum_assured

    # Apply age factor (e.g., increase premium by 2% for each year above 30)
    age_factor = 1 + max(0, (age - 30) * 0.02)

    # Apply health condition multiplier
    health_multiplier = health_multipliers.get(health_condition.lower(), 1.0)

    # Calculate the final premium
    final_premium = base_premium * age_factor * health_multiplier

    return final_premium

In [44]:
# Define the premium calculation function with policy details
def calculate_premium_with_details(policy_name, age, health_condition, sum_assured):
    """
    Calculate the premium for a specific insurance policy based on user inputs and fetch detailed policy information.

    Args:
    policy_name (str): The name of the insurance policy.
    age (int): The age of the potential policyholder.
    health_condition (str): The health condition of the potential policyholder (e.g., "good", "average", "poor").
    sum_assured (float): The sum assured for the insurance policy.

    Returns:
    tuple: A tuple containing the estimated premium amount and detailed policy information.
    """
    # Fetch detailed policy information
    policy_details, _, _ = query_response(f"Provide details about the {policy_name}")

    # Calculate the premium
    estimated_premium = calculate_premium(policy_name, age, health_condition, sum_assured)

    # Print the information for the policy
    print(f"Policy Name: {policy_name}")
    print("#" * len(f"Policy Name: {policy_name}"))  # Underline with #
    print("Estimated Premium:")
    print("------------------")
    print(estimated_premium)
    print("Policy Details:")
    print("---------------")
    print(policy_details)
    print("-" * 80)

    return estimated_premium, policy_details

In [45]:
# Example user inputs
policy_name = "HDFC Life Easy Health Policy"
age = 35
health_condition = "average"
sum_assured = 1000000  # 10,00,000

# Calculate the premium and fetch policy details
estimated_premium, policy_details = calculate_premium_with_details(policy_name, age, health_condition, sum_assured)

Policy Name: HDFC Life Easy Health Policy
#########################################
Estimated Premium:
------------------
13200.0
Policy Details:
---------------
a). Summary: The HDFC Life Easy Health Policy is a health insurance policy offered by HDFC Life Insurance Company Limited.

b). Detailed Explanation:
	The policy covers health-related risks and provides benefits in case of medical emergencies. It includes definitions of terms like Accident, Appointee, Assignee, Authority/IRDAI, Cancellation, Company, Critical Illnesses, and more. The policy allows for assignment and transfer of rights under specific conditions. It also outlines the process for policy cancellation and termination.

c). Key Points:
	- The policy covers health risks and provides benefits for medical emergencies.
	- Definitions of key terms like Accident, Appointee, Assignee, Authority/IRDAI, Cancellation, Company, Critical Illnesses are provided.
	- Assignment and transfer of rights under the policy are allowed u

**Use Case 4**: Policy Comparison Tool
User Story: As a user, I want to compare different insurance policies to understand their benefits, exclusions, and premiums, so that I can make an informed decision.

**Implementation**: Develop a feature that allows users to input multiple policy names and receive a side-by-side comparison of key attributes.

**Rewriting the query_response function to handle the compare policies use case**

In [46]:
def query_response_compare_policies(user_input):
    """
    Generate a response based on user input by querying the query engine and
    retrieving metadata from the source nodes.

    Args:
    user_input (str): The input query provided by the user.

    Returns:
    tuple: A tuple containing the summary, reference, and page number.
    """
    # Extract policy name from the user input
    user_input_lower = user_input.lower()

    # Create a temporary index with the filtered documents
    temp_index = GPTVectorStoreIndex(documents)
    temp_query_engine = temp_index.as_query_engine()

    response = temp_query_engine.query(user_input)

    # Extract the response content
    summary = response.response  # Access the response attribute

    references = []
    page_numbers = []

    # Check if source_nodes attribute exists and if metadata is available
    if hasattr(response, 'source_nodes') and response.source_nodes:
        sources = response.source_nodes  # Access the source_nodes attribute

        # Filter out sources with empty metadata
        sources_with_metadata = [source for source in sources if source.metadata]

        if sources_with_metadata:
            for source in sources_with_metadata:
                references.append(source.metadata.get('file_name', 'Unknown'))
                page_numbers.append(source.metadata.get('page_number', 'Unknown'))

    # Return only the summary, reference, and page number
    reference = references[0] if references else "Unknown"
    page_number = page_numbers[0] if page_numbers else "Unknown"

    return summary, reference, page_number

In [47]:
# Define the compare_policies function
def compare_policies(policy_names):
    """
    Compare different insurance policies based on their benefits, exclusions, and premiums.

    Args:
    policy_names (list): A list of policy names to be compared.

    Returns:
    pd.DataFrame: A DataFrame containing the comparison results for each policy.
    """
    comparison_results = []

    for policy_name in policy_names:
        # Query the RAG system for detailed information about the policy
        benefits_summary, benefits_reference, benefits_page = query_response(f"What are the benefits of the {policy_name}?")
        exclusions_summary, exclusions_reference, exclusions_page = query_response(f"What are the exclusions of the {policy_name}?")
        premiums_summary, premiums_reference, premiums_page = query_response(f"What are the premiums of the {policy_name}?")

        # Store the information for the policy in a dictionary
        policy_info = {
            "Policy Name": policy_name,
            "Benefits": benefits_summary,
            "Exclusions": exclusions_summary,
            "Premiums": premiums_summary
        }

        # Add the policy information to the comparison results
        comparison_results.append(policy_info)

    # Create a DataFrame from the comparison results
    df = pd.DataFrame(comparison_results)

    # Print the DataFrame as a formatted table using tabulate
    print(tabulate(df, headers='keys', tablefmt='pipe'))

    return df

In [48]:
# Example usage
policy_names = [
    "HDFC Life Sanchay Plus",
    "HDFC Life Sampoorna Jeevan",
    "HDFC Life Smart Pension Plan"
]

# Call the comparison function
comparison_results = compare_policies(policy_names)

|    | Policy Name                  | Benefits                                                                                                                                                                                                                                                                   | Exclusions                                                                                                                                                                                                                                                                 | Premiums                                                                                                                                                                                                                               |
|---:|:-----------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------

**Additional improvments on the RAG pipeline using Retriever Router Query Engine**

In [49]:
# Define the policies
policies = [
    {"name": "HDFC Life Easy Health Policy", "type": "health"},
    {"name": "HDFC Life Group Poorna Suraksha Policy", "type": "life"},
    {"name": "HDFC Life Group Term Life Policy", "type": "life"},
    {"name": "HDFC Life Sampoorna Jeevan Plan", "type": "life"},
    {"name": "HDFC Life Sanchay Plus Policy", "type": "life"},
    {"name": "HDFC Life Smart Pension Plan", "type": "pension"},
    {"name": "HDFC Surgicare Plan", "type": "health"},
]

In [50]:
# Create a mapping of policy names to their types
policy_name_to_type = {policy["name"].lower(): policy["type"] for policy in policies}

# Create documents for each policy
documents = [Document(text=policy["name"], metadata=policy) for policy in policies]

# Initialize settings (set chunk size)
Settings.chunk_size = 1024
nodes = Settings.node_parser.get_nodes_from_documents(documents)

# Create indices for each policy type
health_index = VectorStoreIndex([node for node in nodes if node.metadata["type"] == "health"])
life_index = VectorStoreIndex([node for node in nodes if node.metadata["type"] == "life"])
pension_index = VectorStoreIndex([node for node in nodes if node.metadata["type"] == "pension"])

# Create query engines for each index
health_query_engine = health_index.as_query_engine(response_mode="tree_summarize", use_async=True)
life_query_engine = life_index.as_query_engine(response_mode="tree_summarize", use_async=True)
pension_query_engine = pension_index.as_query_engine(response_mode="tree_summarize", use_async=True)

# Create tools for each query engine
health_tool = QueryEngineTool.from_defaults(
    query_engine=health_query_engine,
    description="Useful for questions about health policies."
)
life_tool = QueryEngineTool.from_defaults(
    query_engine=life_query_engine,
    description="Useful for questions about life policies."
)
pension_tool = QueryEngineTool.from_defaults(
    query_engine=pension_query_engine,
    description="Useful for questions about pension policies."
)

In [51]:
# Modifying the query_response function to handle Retriever Router Query Engine
def query_response(user_input):
    """
    Generate a response based on user input by querying the query engine and
    retrieving metadata from the source nodes.

    Args:
    user_input (str): The input query provided by the user.

    Returns:
    tuple: A tuple containing the final response and a list of references.
    """
    # Determine the policy type from the user input
    user_input_lower = user_input.lower()
    policy_type = None
    policy_name = None
    for policy in policies:
        if policy["name"].lower() in user_input_lower:
            policy_type = policy["type"]
            policy_name = policy["name"]
            break

    if not policy_type:
        return "Policy type not found in the query.", []

    # Create the custom prompt with the general context
    custom_prompt = custom_prompt_template.format(
        context=general_context,
        query=user_input,
        summary="{summary}",  # Placeholder to be filled dynamically
        detailed_explanation="{detailed_explanation}",  # Placeholder to be filled dynamically
        key_points="{key_points}"  # Placeholder to be filled dynamically
    )

    # Query the appropriate tool's query engine
    if policy_type == "health":
        response = health_tool.query_engine.query(custom_prompt)
    elif policy_type == "life":
        response = life_tool.query_engine.query(custom_prompt)
    elif policy_type == "pension":
        response = pension_tool.query_engine.query(custom_prompt)
    else:
        return "Policy type not found in the query.", []

    # Extract the response content
    final_response = response.response  # Access the response attribute

    references = []

    # Check if source_nodes attribute exists and if metadata is available
    if hasattr(response, 'source_nodes') and response.source_nodes:
        sources = response.source_nodes  # Access the source_nodes attribute

        # Filter out sources with empty metadata and ensure they match the specific policy name
        sources_with_metadata = [source for source in sources if source.metadata and source.metadata.get('name', '').lower() == policy_name.lower()]

        if sources_with_metadata:
            for source in sources_with_metadata:
                # Ensure 'name' key exists in metadata
                name = source.metadata.get('name', 'Unknown')

                # Add to references list
                references.append(name)

    # Extracting references from the response metadata (if available)
    reference = references[0] if references else "Unknown"

    return final_response.strip(), reference

In [52]:
# A new chatbot with Retriever Router Query Engine
def InsurancePolicyAssistant():
    """
    Simulates a chatbot interaction for querying insurance policy documents.
    """
    print("#####################################################")
    print("# Welcome to the Insurance Policy Document Chatbot! #")
    print("# You can ask questions about the policy documents. #")
    print("# Type 'exit' to end the conversation.              #")
    print("#####################################################")

    conversation_data = []

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == 'exit':
            print("############")
            print("# Goodbye! #")
            print("############")
            break

        # Get the response from the query_response function
        response, reference = query_response(user_input)

        # Print the chatbot response
        print(f"Chatbot: {response}")
        print(f"\nReferences:\nSource: {reference}")
        print("\n" + "-" * 50 + "\n")

        # Append the conversation data
        conversation_data.append({
            "Question": user_input,
            "Answer": response,
            "Reference": reference
        })

    # Convert the conversation data to a DataFrame
    df = pd.DataFrame(conversation_data)

    # Print the DataFrame as a formatted table using tabulate
    print(tabulate(df, headers='keys', tablefmt='pipe'))

In [56]:
# Example usage
InsurancePolicyAssistant()

#####################################################
# Welcome to the Insurance Policy Document Chatbot! #
# You can ask questions about the policy documents. #
# Type 'exit' to end the conversation.              #
#####################################################
You: What are the different fund options available under the HDFC Life Smart Pension Plan, and what are their investment objectives and risk levels?
Chatbot: a) Summary: The HDFC Life Smart Pension Plan offers various fund options with different investment objectives and risk levels.
b) Detailed Explanation: The fund options under the plan may include equity funds, debt funds, balanced funds, and other investment instruments. Each fund has its own investment objective, such as capital appreciation, income generation, or a mix of both. The risk levels associated with these funds can vary, with equity funds generally carrying higher risk but potentially higher returns, while debt funds may offer lower risk but lower return

- We have created a new chatbot **"InsurancePolicyAssistant"** specifically for the custom prompt template.

- The output is now exactly 100% true and the retrievel seems to be faster than any of our implementation before.
